# T13 - 债券指数研究

## 项目概述

本项目是一个完整的债券指数分析系统，主要功能包括：

1. **数据获取**：从MySQL数据库获取债券指数数据和价格数据
2. **指标计算**：计算收益率、最大回撤、年化波动率等核心指标
3. **多周期分析**：支持30天、3个月、半年、1年等多个时间周期的分析
4. **分组统计**：按收益率和久期进行分组，统计各组的平均表现
5. **组合优化**：跨久期进行组合搭配，优化风险收益比
6. **结果展示**：生成综合分析报告和可视化图表

---

## 1. 环境配置

### 1.1 导入依赖包

In [ ]:
# 标准库
import os
import sys
import warnings
import datetime
from pathlib import Path

# 数据处理
import pandas as pd
import numpy as np
from tqdm import tqdm

# 数据库
import sqlalchemy
from sqlalchemy import pool

# 可视化
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

# 忽略警告
warnings.filterwarnings('ignore')

# 打印版本信息
print(f"Python版本: {sys.version}")
print(f"Pandas版本: {pd.__version__}")
print(f"NumPy版本: {np.__version__}")

### 1.2 配置中文字体

In [ ]:
def setup_chinese_font():
    """配置Matplotlib中文字体"""
    font_names = ['WenQuanYi Zen Hei', 'SimHei', 'Microsoft YaHei', 'sans-serif']
    found_font = None
    
    for font_name in font_names:
        try:
            fm.findfont(font_name, fallback_to_default=False)
            found_font = font_name
            print(f"找到可用的中文字体: {found_font}")
            break
        except:
            continue
    
    if found_font:
        plt.rcParams['font.sans-serif'] = [found_font]
        plt.rcParams['axes.unicode_minus'] = False
        print(f"Matplotlib 中文字体已设置为: {found_font}")
    else:
        print("警告: 未找到指定的中文字体，图表中的中文可能无法正确显示")

# 设置中文字体
setup_chinese_font()

# 设置seaborn样式
sns.set_style('whitegrid')

### 1.3 加载配置

In [ ]:
# 导入项目配置
from config import (
    PROJECT_ROOT, DATA_DIR, OUTPUT_DIR, ASSETS_DIR,
    DB_HOST, DB_PORT, DB_NAME, get_database_url,
    RETURN_BIN_SIZE, DURATION_BIN_SIZE, DRAWDOWN_BIN_SIZE,
    TOP_N_INDICES, PERIODS, DEFAULT_PERIOD,
    MIN_COMBO_SIZE, MAX_COMBO_SIZE, SAMPLE_SIZE,
    validate_config
)

# 验证配置
config_errors = validate_config()
if config_errors:
    print("配置警告:")
    for error in config_errors:
        print(f"  - {error}")
else:
    print("配置验证通过!")

print(f"\n项目根目录: {PROJECT_ROOT}")
print(f"数据目录: {DATA_DIR}")
print(f"输出目录: {OUTPUT_DIR}")

---

## 2. 数据获取

### 2.1 数据库连接

In [ ]:
def connect_database():
    """连接到债券数据库"""
    try:
        db_url = get_database_url()
        sql_engine = sqlalchemy.create_engine(
            db_url,
            poolclass=sqlalchemy.pool.NullPool
        )
        cursor = sql_engine.connect()
        print("数据库连接成功")
        return sql_engine, cursor
    except Exception as e:
        print(f"数据库连接失败: {e}")
        return None, None

# 连接数据库
sql_engine, connection = connect_database()

### 2.2 获取债券指数列表

In [ ]:
def get_bond_indices(cursor):
    """获取债券指数代码和名称"""
    query = """
    SELECT trade_code, index_name, duration FROM bond.indexcode
    WHERE index_name LIKE '%%财富%%' AND duration IS NOT NULL 
    AND (index_name LIKE '%%1-3年%%' OR index_name LIKE '%%3-5年%%')
    """
    indices_df = pd.read_sql(query, cursor)
    print(f"获取到 {len(indices_df)} 个债券指数")
    return indices_df

# 获取指数列表
if connection is not None:
    indices_df = get_bond_indices(connection)
    display(indices_df.head(10))
else:
    print("数据库未连接，跳过获取指数列表")
    indices_df = pd.DataFrame()

### 2.3 获取价格数据

In [ ]:
def get_price_data(cursor, trade_code, start_date, end_date):
    """获取指定指数的价格数据"""
    query = f"""
    SELECT DT AS date, CLOSE AS price FROM bond.indexcloseprice
    WHERE TRADE_CODE = '{trade_code}'
    AND DT <= '{end_date}' AND DT >= '{start_date}'
    ORDER BY DT
    """
    df = pd.read_sql(query, cursor)
    if len(df) > 0:
        df['date'] = pd.to_datetime(df['date'])
        df.set_index('date', inplace=True)
    return df

# 计算各周期的开始日期
today = datetime.datetime.now()
end_date = today.strftime('%Y-%m-%d')

period_start_dates = {}
for period_name, days in PERIODS.items():
    start_date = (today - datetime.timedelta(days=days)).strftime('%Y-%m-%d')
    period_start_dates[period_name] = start_date
    print(f"{period_name}: {start_date} ~ {end_date}")

---

## 3. 数据处理

### 3.1 计算基础指标函数

In [ ]:
def calculate_return(prices):
    """计算收益率（百分比）"""
    if len(prices) < 2:
        return None
    return (prices.iloc[-1] / prices.iloc[0] - 1) * 100


def calculate_max_drawdown(prices_series):
    """计算最大回撤（百分比）"""
    if not isinstance(prices_series, pd.Series):
        prices_series = pd.Series(prices_series)
    
    if len(prices_series) < 2:
        return None
    
    prices = prices_series.values
    max_dd = 0.0
    peak = prices[0]
    
    for price in prices:
        if price > peak:
            peak = price
        if peak == 0:
            dd = 0
        else:
            dd = (peak - price) / peak * 100
        max_dd = max(max_dd, dd)
    
    return max_dd


def calculate_volatility(prices_series, annual_factor=252):
    """计算年化波动率（百分比）"""
    if not isinstance(prices_series, pd.Series):
        prices_series = pd.Series(prices_series)
    
    if len(prices_series) < 2:
        return None
    
    daily_returns = prices_series.pct_change().dropna()
    if daily_returns.empty or len(daily_returns) < 2:
        return None
    return daily_returns.std() * np.sqrt(annual_factor) * 100

print("指标计算函数定义完成")

### 3.2 执行数据获取和指标计算

In [ ]:
def analyze_bond_indices(connection, indices_df):
    """执行完整的债券指数分析"""
    if connection is None or indices_df.empty:
        print("数据库未连接或指数列表为空")
        return pd.DataFrame()
    
    today = datetime.datetime.now()
    end_date = today.strftime('%Y-%m-%d')
    
    results = []
    
    for _, index in tqdm(indices_df.iterrows(), total=len(indices_df), desc="分析指数"):
        trade_code = index['trade_code']
        index_name = index['index_name']
        duration = index['duration']
        
        for period_name, days in PERIODS.items():
            start_date = (today - datetime.timedelta(days=days)).strftime('%Y-%m-%d')
            
            # 获取价格数据
            price_data = get_price_data(connection, trade_code, start_date, end_date)
            
            if len(price_data) < 2:
                continue
            
            # 计算指标
            returns = calculate_return(price_data['price'])
            max_dd = calculate_max_drawdown(price_data['price'])
            volatility = calculate_volatility(price_data['price'])
            
            results.append({
                'trade_code': trade_code,
                'index_name': index_name,
                'duration': duration,
                'period': period_name,
                'return': returns,
                'max_drawdown': max_dd,
                'volatility': volatility
            })
    
    results_df = pd.DataFrame(results)
    return results_df

# 执行分析
results_df = analyze_bond_indices(connection, indices_df)

if not results_df.empty:
    print(f"\n分析完成，共 {len(results_df)} 条记录")
    display(results_df.head(10))
    
    # 保存结果
    output_file = OUTPUT_DIR / 'bond_indices_results.csv'
    results_df.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"\n结果已保存至: {output_file}")
else:
    print("未获取到任何分析结果")

### 3.3 数据分组分析

In [ ]:
def create_bins(data, column, bin_size):
    """创建分组区间"""
    if len(data) == 0:
        return np.array([0, bin_size])
    
    min_val = data[column].min() - (data[column].min() % bin_size)
    max_val = data[column].max() + bin_size - (data[column].max() % bin_size)
    return np.arange(min_val, max_val + bin_size/2, bin_size)


def group_analysis(results_df, group_by='return', bin_size=0.5):
    """按指定字段进行分组分析"""
    if results_df.empty:
        return pd.DataFrame(), results_df
    
    if group_by == 'return':
        bins = create_bins(results_df, 'return', bin_size)
        column = 'return'
    else:
        bins = create_bins(results_df, 'duration', bin_size)
        column = 'duration'
    
    # 创建分组标签
    labels = [f"{bins[i]:.1f}-{bins[i+1]:.1f}" for i in range(len(bins)-1)]
    
    # 分组
    results_df['group'] = pd.cut(results_df[column], bins=bins, labels=labels)
    
    # 按分组进行统计
    grouped_stats = results_df.groupby(['group', 'period']).agg({
        'return': 'mean',
        'max_drawdown': 'mean',
        'volatility': 'mean',
        'trade_code': 'count'
    }).rename(columns={'trade_code': 'count'})
    
    return grouped_stats.reset_index(), results_df

# 执行分组分析
if not results_df.empty:
    # 收益率分组
    return_groups, results_df = group_analysis(results_df, group_by='return', bin_size=RETURN_BIN_SIZE)
    print("收益率分组分析:")
    display(return_groups.head(10))
    
    # 久期分组
    duration_groups, results_df = group_analysis(results_df, group_by='duration', bin_size=DURATION_BIN_SIZE)
    print("\n久期分组分析:")
    display(duration_groups.head(10))
    
    # 保存分组结果
    return_groups.to_csv(OUTPUT_DIR / 'return_group_analysis.csv', index=False, encoding='utf-8-sig')
    duration_groups.to_csv(OUTPUT_DIR / 'duration_group_analysis.csv', index=False, encoding='utf-8-sig')
    print(f"\n分组分析结果已保存至 {OUTPUT_DIR}")

---

## 4. 核心逻辑

### 4.1 三维分组分析

In [ ]:
def three_dimensional_analysis(results_df, period='1年'):
    """生成收益-久期-最大回撤三维分组表"""
    period_data = results_df[results_df['period'] == period].copy()
    
    if len(period_data) == 0:
        print(f"警告: {period}期间没有数据")
        return pd.DataFrame(), pd.DataFrame()
    
    # 创建收益率分组
    return_bins = create_bins(period_data, 'return', RETURN_BIN_SIZE)
    return_labels = [f"{return_bins[i]:.1f}%-{return_bins[i+1]:.1f}%" for i in range(len(return_bins)-1)]
    period_data['return_group'] = pd.cut(period_data['return'], bins=return_bins, labels=return_labels)
    
    # 创建久期分组
    duration_bins = create_bins(period_data, 'duration', DURATION_BIN_SIZE)
    duration_labels = [f"{duration_bins[i]:.1f}-{duration_bins[i+1]:.1f}" for i in range(len(duration_bins)-1)]
    period_data['duration_group'] = pd.cut(period_data['duration'], bins=duration_bins, labels=duration_labels)
    
    # 创建最大回撤分组
    drawdown_bins = create_bins(period_data, 'max_drawdown', DRAWDOWN_BIN_SIZE)
    drawdown_labels = [f"{drawdown_bins[i]:.1f}%-{drawdown_bins[i+1]:.1f}%" for i in range(len(drawdown_bins)-1)]
    period_data['drawdown_group'] = pd.cut(period_data['max_drawdown'], bins=drawdown_bins, labels=drawdown_labels)
    
    # 创建三维分组统计
    three_dim_stats = period_data.groupby(['return_group', 'duration_group', 'drawdown_group']).size().reset_index(name='count')
    
    # 转换为透视表格式
    pivot_table = three_dim_stats.pivot_table(
        index=['return_group', 'duration_group'],
        columns='drawdown_group',
        values='count',
        fill_value=0
    )
    
    return pivot_table, three_dim_stats

# 执行三维分组分析
if not results_df.empty:
    for period in PERIODS.keys():
        print(f"\n=== {period} 三维分组分析 ===")
        pivot_table, three_dim_stats = three_dimensional_analysis(results_df, period)
        
        if not pivot_table.empty:
            display(pivot_table.head())
            
            # 保存结果
            pivot_table.to_csv(OUTPUT_DIR / f'three_dim_analysis_{period}.csv', encoding='utf-8-sig')
            print(f"结果已保存至 {OUTPUT_DIR / f'three_dim_analysis_{period}.csv'}")

### 4.2 投资组合优化

In [ ]:
import itertools

def optimize_portfolio(results_df, indices_df, period='1年', max_combinations=10000):
    """跨久期分组进行组合优化"""
    period_data = results_df[results_df['period'] == period].copy()
    
    if len(period_data) == 0:
        print(f"警告: {period}期间没有数据")
        return {}
    
    # 按久期分组
    duration_bins = create_bins(period_data, 'duration', DURATION_BIN_SIZE)
    duration_labels = [f"{duration_bins[i]:.1f}-{duration_bins[i+1]:.1f}" for i in range(len(duration_bins)-1)]
    period_data['duration_group'] = pd.cut(period_data['duration'], bins=duration_bins, labels=duration_labels)
    
    # 计算每个久期组的平均指标
    duration_group_stats = period_data.groupby('duration_group').agg({
        'return': 'mean',
        'max_drawdown': 'mean'
    })
    
    best_combinations = {}
    
    for duration_group in duration_labels:
        if duration_group not in period_data['duration_group'].values:
            continue
        
        group_indices = period_data[period_data['duration_group'] == duration_group]
        
        if len(group_indices) <= 1:
            best_combinations[duration_group] = {
                'indices': group_indices['trade_code'].tolist(),
                'names': group_indices['index_name'].tolist(),
                'return': group_indices['return'].mean(),
                'max_drawdown': group_indices['max_drawdown'].mean()
            }
            continue
        
        # 获取目标值
        target_return = duration_group_stats.loc[duration_group, 'return']
        target_drawdown = duration_group_stats.loc[duration_group, 'max_drawdown']
        
        best_combo = None
        best_score = -float('inf')
        
        all_codes = group_indices['trade_code'].unique()
        min_combo_size = min(MIN_COMBO_SIZE, len(all_codes))
        max_combo_size = min(MAX_COMBO_SIZE, len(all_codes))
        
        for combo_size in range(min_combo_size, max_combo_size + 1):
            combos = list(itertools.combinations(range(len(all_codes)), combo_size))
            np.random.shuffle(combos)
            combos = combos[:max_combinations // (max_combo_size - min_combo_size + 1)]
            
            for combo in combos:
                selected_codes = [all_codes[i] for i in combo]
                combo_data = group_indices[group_indices['trade_code'].isin(selected_codes)]
                
                combo_return = combo_data['return'].mean()
                combo_drawdown = combo_data['max_drawdown'].mean()
                
                if combo_return > target_return and combo_drawdown < target_drawdown:
                    score = (combo_return - target_return) / target_return - (target_drawdown - combo_drawdown) / target_drawdown
                    
                    if best_combo is None or score > best_score:
                        best_combo = selected_codes
                        best_score = score
                        best_return = combo_return
                        best_drawdown = combo_drawdown
        
        if best_combo:
            best_names = indices_df[indices_df['trade_code'].isin(best_combo)]['index_name'].tolist()
            best_combinations[duration_group] = {
                'indices': best_combo,
                'names': best_names,
                'return': best_return,
                'max_drawdown': best_drawdown,
                'target_return': target_return,
                'target_drawdown': target_drawdown
            }
    
    return best_combinations

# 执行组合优化
if not results_df.empty:
    for period in PERIODS.keys():
        print(f"\n=== {period} 投资组合优化 ===")
        portfolio_results = optimize_portfolio(results_df, indices_df, period=period)
        
        if portfolio_results:
            # 转换为DataFrame
            portfolio_df = pd.DataFrame([
                {
                    '久期分组': group,
                    '指数代码': ','.join(data['indices']),
                    '指数名称': ','.join(data['names']),
                    '组合收益': data['return'],
                    '组合最大回撤': data['max_drawdown'],
                    '目标收益': data.get('target_return', ''),
                    '目标最大回撤': data.get('target_drawdown', '')
                }
                for group, data in portfolio_results.items()
            ])
            
            display(portfolio_df)
            
            # 保存结果
            portfolio_df.to_csv(OUTPUT_DIR / f'portfolio_optimization_{period}.csv', index=False, encoding='utf-8-sig')
            print(f"结果已保存至 {OUTPUT_DIR / f'portfolio_optimization_{period}.csv'}")

---

## 5. 执行与测试

### 5.1 可视化 - 收益率与风险分析

In [ ]:
def plot_return_distribution(results_df, period='1年'):
    """绘制收益率分布图"""
    period_data = results_df[results_df['period'] == period].copy()
    
    if len(period_data) == 0:
        print(f"警告: {period}期间没有数据")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 收益率分布
    axes[0, 0].hist(period_data['return'].dropna(), bins=20, alpha=0.7, color='steelblue')
    axes[0, 0].set_xlabel('收益率 (%)')
    axes[0, 0].set_ylabel('数量')
    axes[0, 0].set_title(f'{period} 收益率分布')
    axes[0, 0].axvline(period_data['return'].mean(), color='red', linestyle='--',
                        label=f'平均: {period_data["return"].mean():.2f}%')
    axes[0, 0].legend()
    
    # 收益率 vs 波动率
    axes[0, 1].scatter(period_data['volatility'], period_data['return'], alpha=0.6, c='steelblue')
    axes[0, 1].set_xlabel('年化波动率 (%)')
    axes[0, 1].set_ylabel('收益率 (%)')
    axes[0, 1].set_title('收益率 vs 波动率')
    axes[0, 1].grid(True, linestyle='--', alpha=0.7)
    
    # 收益率 vs 最大回撤
    axes[1, 0].scatter(period_data['max_drawdown'], period_data['return'], alpha=0.6, c='steelblue')
    axes[1, 0].set_xlabel('最大回撤 (%)')
    axes[1, 0].set_ylabel('收益率 (%)')
    axes[1, 0].set_title('收益率 vs 最大回撤')
    axes[1, 0].grid(True, linestyle='--', alpha=0.7)
    
    # 久期 vs 收益率
    axes[1, 1].scatter(period_data['duration'], period_data['return'], alpha=0.6, c='steelblue')
    axes[1, 1].set_xlabel('久期 (年)')
    axes[1, 1].set_ylabel('收益率 (%)')
    axes[1, 1].set_title('久期 vs 收益率')
    axes[1, 1].grid(True, linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    
    # 保存图表
    output_path = OUTPUT_DIR / f'bond_index_analysis_{period}.png'
    plt.savefig(output_path, dpi=100, bbox_inches='tight')
    print(f"图表已保存至: {output_path}")
    
    plt.show()

# 执行可视化
if not results_df.empty:
    for period in PERIODS.keys():
        print(f"\n=== {period} 可视化分析 ===")
        plot_return_distribution(results_df, period)

### 5.2 可视化 - 相关性热图

In [ ]:
def plot_correlation_matrix(results_df, period='1年'):
    """绘制指标相关性矩阵"""
    period_data = results_df[results_df['period'] == period].copy()
    
    if len(period_data) == 0:
        print(f"警告: {period}期间没有数据")
        return
    
    # 计算相关性矩阵
    corr_matrix = period_data[['return', 'max_drawdown', 'volatility', 'duration']].corr()
    
    # 绘制热图
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, ax=ax)
    ax.set_title(f'{period} 指标相关性矩阵', fontsize=14)
    
    plt.tight_layout()
    
    # 保存图表
    output_path = OUTPUT_DIR / f'correlation_matrix_{period}.png'
    plt.savefig(output_path, dpi=100, bbox_inches='tight')
    print(f"图表已保存至: {output_path}")
    
    plt.show()

# 执行相关性分析
if not results_df.empty:
    plot_correlation_matrix(results_df, DEFAULT_PERIOD)

### 5.3 可视化 - 三维分组热图

In [ ]:
def plot_three_dim_heatmap(results_df, period='1年'):
    """绘制三维分组热图"""
    pivot_table, _ = three_dimensional_analysis(results_df, period)
    
    if pivot_table.empty:
        print(f"警告: {period}期间没有数据")
        return
    
    fig, ax = plt.subplots(figsize=(14, 10))
    sns.heatmap(pivot_table, annot=True, cmap='YlGnBu', fmt='g', linewidths=0.5, ax=ax)
    ax.set_title(f'{period} 收益-久期-最大回撤三维分组热图', fontsize=14)
    
    plt.tight_layout()
    
    # 保存图表
    output_path = OUTPUT_DIR / f'three_dim_heatmap_{period}.png'
    plt.savefig(output_path, dpi=100, bbox_inches='tight')
    print(f"图表已保存至: {output_path}")
    
    plt.show()

# 执行三维热图可视化
if not results_df.empty:
    plot_three_dim_heatmap(results_df, DEFAULT_PERIOD)

---

## 6. 工具函数

### 6.1 生成分析报告

In [ ]:
def generate_summary_report(results_df, period='1年'):
    """生成摘要报告"""
    period_data = results_df[results_df['period'] == period].copy()
    
    if len(period_data) == 0:
        print(f"警告: {period}期间没有数据")
        return
    
    # 计算统计信息
    stats = {
        'total_indices': len(period_data['trade_code'].unique()),
        'avg_return': period_data['return'].mean(),
        'avg_drawdown': period_data['max_drawdown'].mean(),
        'avg_volatility': period_data['volatility'].mean(),
        'avg_duration': period_data['duration'].mean(),
        'max_return': period_data['return'].max(),
        'min_drawdown': period_data['max_drawdown'].min()
    }
    
    print(f"\n{'='*60}")
    print(f"债券指数分析报告 - {period}")
    print(f"{'='*60}")
    print(f"指数总数: {stats['total_indices']}")
    print(f"平均收益率: {stats['avg_return']:.2f}%")
    print(f"平均最大回撤: {stats['avg_drawdown']:.2f}%")
    print(f"平均年化波动率: {stats['avg_volatility']:.2f}%")
    print(f"平均久期: {stats['avg_duration']:.2f}年")
    print(f"最高收益率: {stats['max_return']:.2f}%")
    print(f"最低最大回撤: {stats['min_drawdown']:.2f}%")
    print(f"{'='*60}")
    
    # 收益率TOP5
    print(f"\n收益率TOP5指数:")
    top_return = period_data.nlargest(5, 'return')[['index_name', 'return', 'max_drawdown', 'duration']]
    display(top_return)
    
    # 最大回撤最低TOP5
    print(f"\n最大回撤最低TOP5指数:")
    low_drawdown = period_data.nsmallest(5, 'max_drawdown')[['index_name', 'return', 'max_drawdown', 'duration']]
    display(low_drawdown)
    
    return stats

# 生成报告
if not results_df.empty:
    stats = generate_summary_report(results_df, DEFAULT_PERIOD)

### 6.2 关闭数据库连接

In [ ]:
# 关闭数据库连接
if connection is not None:
    connection.close()
    print("数据库连接已关闭")

---

## 总结

本项目完成了一个完整的债券指数分析系统，主要功能包括：

1. **数据获取**：从MySQL数据库获取债券指数列表和历史价格数据
2. **指标计算**：计算收益率、最大回撤、年化波动率等核心指标
3. **分组分析**：按收益率和久期进行分组统计
4. **三维分析**：收益-久期-最大回撤三维分组分析
5. **组合优化**：跨久期分组进行投资组合优化
6. **可视化展示**：生成多种分析图表

所有分析结果已保存至输出目录。